# Fase 2 — Consolidación: de las fuentes a un máster por partido

**TFM · Modelo predictivo 1X2 de LaLiga**

La fase anterior (**`01_EXTRACCION_DE_FUENTES.ipynb`**) dejó las tablas limpias en
`data/02_processed/`, cada una con su propia forma y su propia manera de nombrar equipos y
partidos. Este notebook documenta la **segunda fase**: **unir todas esas tablas en una única
tabla maestra con una fila por partido de LaLiga**, sobre la que después se construyen las
variables del modelo.

No se vuelve a describir *qué* contiene cada fuente —eso ya está en la Fase 1—. Aquí el foco es
**cómo se hace la unión**: el orden que seguimos, los **problemas que van apareciendo** (rejillas
distintas, fechas que no cuadran, formatos largos, claves de dos mundos, huecos que no se deben
rellenar con ceros) y **cómo los resolvemos**. Cada paso termina *comprobando la unión*: que
sigue habiendo una fila por partido, que no se ha multiplicado ninguna fila y qué cobertura
aporta cada fuente.

El trabajo pesado vive en `scripts/consolidate/*.py` (funciones puras, testeables e
idempotentes, igual que los extractores). El notebook las **llama y verifica**; no reimplementa
la lógica, para que lo que se documenta aquí sea exactamente lo que corre en producción.

---

## 1 · El problema de fondo: rejillas distintas y dos "mundos" de clave

Unir las fuentes no es un simple `concat`. Cada tabla vive en una **granularidad** distinta y el
partido se identifica de maneras diferentes. Antes de tocar nada conviene ver el mapa completo:

| Fuente | Granularidad de partida | ¿Cómo identifica el partido? |
|---|---|---|
| football-data | **1 fila / partido** | `match_key` (fecha + equipos) |
| Understat | 1 fila / partido | `match_key` (pero **su fecha puede desfasar 1-2 días**) |
| Sofascore | 1 fila / partido | `match_key` **y** `match_id` |
| ClubElo | **1 fila / club × intervalo de validez** | `team_key` + rango de fechas (¡no hay partido!) |
| Valor de plantilla | **1 fila / club × temporada** | `season` + `team_key` |
| Valor por jugador | 1 fila / jugador × temporada | `season` + `team_key` + jugador |
| Fatiga | **1 fila / equipo × evento** (multi-competición) | `event_id` |
| Disponibilidad | **1 fila / equipo × partido × lado** | `event_id` + `side` |
| Alineaciones | **1 fila / equipo × evento** | `event_id` + `side` |
| Calidad de ocasión | **1 fila / equipo × partido × lado** | `match_key` + `side` (no pasa por `event_id`) |
| Minutos por jugador | **1 fila / jugador × evento** | `event_id` + `side` + `player_key` |

La última fila es la única fuente que **no llega nunca al máster**: los minutos por jugador
viven un nivel por debajo del partido y no hay forma honesta de aplanarlos a una fila sin
perder justo lo que los hace útiles. Entran como **materia prima** de la sección 11, donde se
cruzan contra las bajas para responder qué cuota de los minutos de su equipo venía disputando
cada ausente. Lo que viaja al máster es esa respuesta, no la tabla.

Aparecen **dos mundos de clave** que hay que reconciliar:

- El **mundo `match_key`** — la cadena canónica `AAAA-MM-DD|local|visitante` que produce
  `teamnames.py`. Lo hablan football-data, Understat y Sofascore.
- El **mundo `event_id`** — el identificador interno de la familia Sofascore (fatiga,
  disponibilidad, bajas, alineaciones y minutos). No sabe nada de `match_key`.

**La estrategia de consolidación** se deduce de esa tabla:
1. Elegir una fuente **ya a nivel de partido y con el target** como **columna vertebral** (*spine*):
   football-data.
2. Traducir todo lo demás a **una fila por `match_key`** antes de unirlo (los formatos largos y de
   temporada hay que *reducirlos*; el mundo `event_id` hay que *puentearlo* a `match_key`).
3. Unir cada pieza al spine con un **left-merge por `match_key`**, comprobando en cada paso que no
   se multiplica ninguna fila.

Las secciones siguientes recorren ese plan pieza a pieza.

## 2 · Preparación

Localizamos la raíz del repo, damos de alta en el `path` los módulos de consolidación
(`scripts/consolidate/`) y de extracción (`scripts/extractors/`, de donde se reutiliza
`teamnames`), y cargamos las tablas de fuente. Tres de ellas --alineaciones, minutos y calidad de ocasión-- se leen
como opcionales: proceden de volcados propios y largos, de una petición por partido, y el
máster se construye igual sin ellas. Definimos también un ayudante, **`comprobar()`**, que
usaremos tras cada unión para verificar de un vistazo la **forma**, la **unicidad de `match_key`**
(una fila por partido) y, opcionalmente, que **no ha habido fan-out** (que el nº de filas no ha
crecido respecto al spine).

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)

def _find_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts" / "consolidate").is_dir():
            return cand
    raise FileNotFoundError("No encuentro la raíz del repo desde " + str(p))

ROOT = _find_root()
for sub in ("scripts/consolidate", "scripts/extractors"):
    d = str(ROOT / sub)
    if d not in sys.path:
        sys.path.insert(0, d)
PROC = ROOT / "data" / "02_processed"
print("Repo:", ROOT)

# módulos de consolidación (la lógica real; el notebook solo la orquesta y verifica)
from build_master import _load, build_master
from spine import build_spine, add_understat, add_clubelo
from reducers import (event_to_match_key, reduce_sofascore, reduce_fatigue,
                      reduce_availability, reduce_lineups, reduce_shot_quality)
from squadvalue import reduce_squad_value
from absences import build_absences

def comprobar(df, nombre, esperado=None, clave="match_key"):
    """Chequeo rápido de una unión: forma + clave única + (opcional) sin fan-out."""
    unico = df[clave].is_unique if clave in df.columns else None
    msg = f"[{nombre}]  {len(df):,} filas × {df.shape[1]} cols"
    if unico is not None:
        msg += f"  |  {clave} único: {unico}"
    if esperado is not None:
        msg += f"  |  == {esperado:,} partidos del spine: {len(df) == esperado}"
    print(msg)
    return df

data = _load()   # tablas procesadas + el raw de bajas
print("Tablas cargadas:", ", ".join(k for k, v in data.items() if v is not None))

# Alineaciones, minutos y calidad de ocasion se leen como OPCIONALES: nacen de
# volcados propios y largos (`extractors/lineups.py` y
# `extractors/understat_shots.py`, una peticion por partido cada uno) y el master
# se construye igual sin ellos, con sus bloques en NaN. Que la cadena no dependa
# de un backfill de horas es una decision de diseno, no una concesion.
faltan = [k for k, v in data.items() if v is None]
print("Opcionales ausentes:", faltan or "ninguno")

Repo: C:\Users\admin\Desktop\TFM 2026\TFM\repos\laliga-predictor-engine

Tablas cargadas: fd, us, elo, sofa, fat, av, tv, pv, missing


## 3 · La clave universal: `match_key` y el puente `event_id → match_key`

Toda la unión se apoya en **una única clave canónica de partido**. `teamnames.py` reduce cualquier
variante de nombre (`"Athletic Club"`, `"Bilbao"`, `"ath-bilbao"`) a una **clave de equipo**
estable (`athletic_bilbao`) y compone el **`match_key`** `AAAA-MM-DD|local|visitante`. Esa cadena
es el pegamento entre las fuentes del *mundo match_key*.

Para el *mundo event_id* (fatiga, disponibilidad, bajas) necesitamos un **puente**. La tabla de
Sofascore es la única que trae **a la vez** `match_id` (== `event_id`) **y** `match_key`, así que
de ella extraemos el diccionario `event_id → match_key`. Es la pieza que permite que una fila de
fatiga, identificada por `event_id`, sepa a qué `match_key` del máster pertenece.

Comprobamos que ese puente es **uno-a-uno** (si `match_id` estuviera duplicado, el mapeo sería
ambiguo y el extractor falla ruidosamente).

In [2]:
e2k = event_to_match_key(data["sofa"])   # Series: match_id -> match_key
print("Puente event_id -> match_key")
print("  eventos mapeados :", e2k.notna().sum())
print("  match_id único   :", e2k.index.is_unique)
print("  ejemplo          :", e2k.index[0], "->", e2k.iloc[0])

Puente event_id -> match_key
  eventos mapeados : 3800
  match_id único   : True
  ejemplo          : 7125608 -> 2016-08-19|malaga|osasuna


## 4 · El *spine*: football-data como columna vertebral

El máster se construye alrededor de un esqueleto que define **qué partidos existen**. Ese esqueleto
es **football-data**: ya viene a una fila por partido, trae el **target `ftr`** y todas las cuotas,
y su `match_key` es la referencia contra la que se alinea todo lo demás.

El spine se enriquece con las dos fuentes históricas que también van por partido —**Understat**
(xG) y **ClubElo** (fuerza)— mediante la función `build_spine()`. Las secciones 5 y 6 abren esa
caja para ver los dos problemas que resuelve por dentro; aquí primero fijamos el esqueleto y
guardamos su tamaño (`N_PARTIDOS`), que será la vara de medir del resto del notebook: **ninguna
unión posterior debe cambiarlo**.

In [3]:
fd = data["fd"]
base = fd[fd["season_year"] >= 2016].reset_index(drop=True)
N_PARTIDOS = len(base)
comprobar(base, "spine base (solo football-data)")
seas = sorted(base["season"].dropna().unique())
print("temporadas:", seas[0], "->", seas[-1], f"({len(seas)} temporadas)")
print("target ftr:", base["ftr"].value_counts(normalize=True).round(3).to_dict())

[spine base (solo football-data)]  3,800 filas × 50 cols  |  match_key único: True
temporadas: 2016/17 -> 2025/26 (10 temporadas)
target ftr: {'H': 0.455, 'A': 0.283, 'D': 0.262}


## 5 · Understat: mismo partido, fecha distinta → *fallback* ±2 días

Understat ya viene reducido a una fila por partido y con `match_key`, así que en teoría bastaría un
merge por esa clave. **El problema:** la fecha de un partido en Understat y en football-data no
siempre coincide —a veces difiere en **1 o 2 días** (husos horarios, partidos a caballo de la
medianoche, correcciones de calendario)—. Y como el `match_key` **incluye la fecha**, cuando la
fecha baila el `match_key` deja de casar y el partido se quedaría **sin xG**.

**La solución** es un merge en dos pasos (`spine._prekeyed_tolerant`): primero el `match_key`
**exacto**; para los que quedan sin casar, se busca en Understat la fila con los **mismos equipos**
(`home_key`, `away_key`) cuya fecha esté a **±2 días** y se toma la más cercana. Lo comprobamos
midiendo cuánto casa el exacto y cuánto sube tras el *fallback*.

In [5]:
# nº de partidos que casan por match_key EXACTO (antes del fallback)
exact = base[["match_key"]].merge(
    data["us"][["match_key", "us_home_xg"]].drop_duplicates("match_key"),
    on="match_key", how="left")
n_exact = exact["us_home_xg"].notna().sum()

# la función real aplica exacto + fallback ±2d y renombra us_home_* -> home_*
spine_us = add_understat(base, data["us"])
comprobar(spine_us, "spine + Understat", esperado=N_PARTIDOS)

cob = 1 - spine_us["home_xg"].isna().mean()
print(f"casan por match_key exacto : {n_exact:,}/{N_PARTIDOS:,} = {n_exact/N_PARTIDOS:.1%}")
print(f"cobertura xG tras fallback : {cob:.1%}  (recuperados por ±2d: {int(cob*N_PARTIDOS)-n_exact})")
print("columnas xG añadidas       :", [c for c in spine_us.columns if c in
      ("home_xg","away_xg","home_npxg","away_npxg","home_ppda","away_ppda","home_deep","away_deep","home_xpts","away_xpts")])

[spine + Understat]  3,800 filas × 60 cols  |  match_key único: True  |  == 3,800 partidos del spine: True
casan por match_key exacto : 3,782/3,800 = 99.5%
cobertura xG tras fallback : 100.0%  (recuperados por ±2d: 18)
columnas xG añadidas       : ['home_xg', 'away_xg', 'home_npxg', 'away_npxg', 'home_ppda', 'away_ppda', 'home_deep', 'away_deep', 'home_xpts', 'away_xpts']


## 6 · ClubElo: de intervalos a un Elo por partido (`merge_asof`) y de nivel a fuerza relativa

ClubElo **no está a nivel de partido**: es un histórico en **formato largo por intervalos de
validez** —una fila por club y tramo `from`→`to` en el que su Elo se mantiene constante—. No se
puede unir por `match_key` porque no hay partido; hay que preguntar, para cada equipo, **cuál era
su Elo el día del partido**.

**La solución al primer problema** es un `pd.merge_asof` con `direction="backward"` por `team_key`
sobre la fecha: para cada equipo del spine toma el **último tramo de Elo iniciado en o antes** de
la fecha del partido. Se hace dos veces (local y visitante) y se añade
`elo_diff = home_elo - away_elo`. Como `clubelo_elo.parquet` ya trae `team_key` normalizado, no
hace falta remapear nombres. La cobertura esperada es ~100 % (todos los clubes de LaLiga están en
ClubElo).

**Hay un segundo problema, y es el que decide la forma en que el Elo llega al modelo.** La serie
que publica el proyecto es el **empalme de dos vías de la misma fuente**: el archivo de la ruta de
descarga jubilada hasta el 18-10-2022 y las fichas por club de `clubelo.com/ESP` desde entonces.
ClubElo rebasó su escala en ese corte —la vía web va unos **85 puntos por encima** y **comprime la
dispersión un 11 %**—, así que los niveles crudos de los dos tramos no son comparables entre sí.
En nivel, el empalme sería un **escalón de fuerza simultáneo para los veinte clubes**, y el modelo
lo leería como un acontecimiento deportivo.

Reconciliar los niveles no es posible: el mejor ajuste afín sobre el solape
(`web = 403.6 + 0.8053 · api`, R² 0.965) deja un residuo de 18.7 puntos de desviación, del orden
de lo que separa a dos equipos de mitad de tabla, y reajustarlo semana a semana no lo arregla. Lo
que **sí** se reconcilia es la **posición relativa**: restar la media de la jornada y dividir por su
desviación deshace exactamente una transformación afín. Por eso `add_clubelo` emite, además del
nivel, el **par tipificado dentro de la jornada** `home_elo_z` / `away_elo_z` —cuántas desviaciones
separan al club de la media de la liga ese día—, y es ese par, no el nivel, lo que consume el
catálogo de variables.

Dos detalles que no son cosméticos:

- **La base de normalización son los veinte clubes de Primera de esa temporada**, según el censo
  que el propio proyecto mantiene (`spine.censo_por_temporada`), nunca el campo `level` de
  ClubElo ni los dos equipos del partido. Tipificar sobre una base distinta en entrenamiento y en
  inferencia daría dos cifras distintas para el mismo partido, y por eso la ruta de inferencia
  puede **imponer** el censo con el argumento `censo=`.
- **La tipificación vive en la consolidación y no en el extractor**, porque la base son los clubes
  que juegan Primera en cada temporada y eso solo lo sabe la capa que tiene el universo de
  partidos delante.

El nivel crudo se queda en el máster como **columna descriptiva y auditable** —sirve para mostrar
el número y para comprobar la unión— y viajan con él `elo_via` y `elo_base_n`, el rastro de
auditoría del empalme: de qué vía sale el Elo de esa fecha y con cuántos clubes se tipificó. Quien
vigila el precio del empalme partido a partido es la séptima invariante de `qa_master`.


In [ ]:
spine = add_clubelo(spine_us, data["elo"])   # spine completo: fd + xG + Elo
comprobar(spine, "spine + ClubElo (spine final)", esperado=N_PARTIDOS)
print("cobertura Elo  : home=%.1f%%  away=%.1f%%" % (
    100*(1-spine["home_elo"].isna().mean()), 100*(1-spine["away_elo"].isna().mean())))

# Rastro del empalme: cuantos partidos resuelve cada via y sobre cuantos clubes se
# tipifico. `elo_base_n` debe valer 20 en todas las filas -- la base son los veinte
# de Primera de esa temporada, y una base mas corta significaria censo incompleto.
print("partidos por via:", spine["elo_via"].value_counts().to_dict())
print("clubes de la base de tipificacion:", sorted(spine["elo_base_n"].dropna().unique()))

# Y la comprobacion de que la tipificacion hace lo que dice: apilados los dos lados,
# la fuerza relativa tiene media ~0 y desviacion 1 por construccion.
z = pd.concat([spine["home_elo_z"], spine["away_elo_z"]]).dropna()
print("elo_z  n=%d  media=%+.4f  sd=%.4f  min=%.3f  max=%.3f"
      % (len(z), z.mean(), z.std(ddof=0), z.min(), z.max()))

print("\nMuestra del spine:")
display(spine[["match_key","date","ftr","home_xg","away_xg",
               "home_elo","away_elo","elo_diff","home_elo_z","away_elo_z",
               "elo_via"]].head(4))

# equivalente a las 3 celdas anteriores en una sola llamada de produccion:
assert build_spine(data["fd"], data["us"], data["elo"]).shape == spine.shape


## 7 · Sofascore: cruzar del *mundo event_id* al *mundo match_key*

Sofascore ya viene a una fila por partido con `match_key`, así que **no hay que reducir nada**:
`reduce_sofascore()` se limita a seleccionar las ~104 columnas `sofa_*` + los dos ratings medios
(renombrados a `home_sofa_rating` / `away_sofa_rating`) y a garantizar `match_key` único.

Su papel adicional es doble: **(a)** aporta el grueso de la estadística de juego, y **(b)** es la
tabla que sostiene el **puente `event_id → match_key`** de la sección 3.

In [9]:
r_sofa = reduce_sofascore(data["sofa"])
comprobar(r_sofa, "Sofascore reducido")
print("jornada (round):", sorted(r_sofa["round"].dropna().unique())[:3], "...",
      sorted(r_sofa["round"].dropna().unique())[-2:])
print("columnas sofa_*:", sum(col.startswith("sofa_") for col in r_sofa.columns),
      "| ratings:", [col for col in r_sofa.columns if col.endswith("_sofa_rating")])
display(r_sofa[["match_key","round","home_sofa_rating","away_sofa_rating"]].head(3))

[Sofascore reducido]  3,800 filas × 111 cols  |  match_key único: True
jornada (round): [1, 2, 3] ... [37, 38]
columnas sofa_*: 104 | ratings: ['home_sofa_rating', 'away_sofa_rating']


,match_key,round,home_sofa_rating,away_sofa_rating
0,2016-08-19|malaga|osasuna,1,6.735714,6.885714
1,2016-08-19|deportivo_la_coruna|eibar,1,6.835714,6.785714
2,2016-08-20|barcelona|real_betis,1,7.392308,6.335714


## 8 · Fatiga: formato largo, multi-competición y partidos del futuro

La fatiga es la fuente **más lejana** a "una fila por partido". Trae tres complicaciones a la vez:

- Está en **formato largo**: una fila por **equipo y evento**, no por partido. Hay que **pivotar**
  los dos lados a columnas `home_fat_*` / `away_fat_*`.
- Es **multi-competición**: incluye Copa, Europa y selecciones. Eso es *deseado* —las señales
  (`rest_days`, `games_last_7/14`, ...) miden congestión de **todas** las competiciones—, pero solo
  las filas de **`competition == "LaLiga"`** definen partidos del máster.
- Contiene **eventos futuros** (fixtures hasta 2027) que **no existen** en Sofascore. Al mapearlos
  con el puente `event_id → match_key` quedan `NaN` y **se descartan**: no son parte del universo.

`reduce_fatigue()` hace justo eso. Comprobamos que el resultado vuelve a ser una fila por
`match_key` y que los eventos futuros/no casados han caído.

In [ ]:
r_fat = reduce_fatigue(data["fat"], e2k)
comprobar(r_fat, "Fatiga reducida (home/away)")
n_laliga = (data["fat"]["competition"] == "LaLiga").sum()
print(f"filas fatiga totales: {len(data['fat']):,}  |  solo LaLiga: {n_laliga:,}"
      f"  |  partidos tras pivotar+puentear: {len(r_fat):,}")
# La cobertura se mide en la direccion util: que fraccion del SPINE queda cubierta
# por la fuente. La direccion contraria (filas de la fuente que existen en el spine)
# es 1.0 por construccion y no informa de nada.
print("cobertura del spine:", f"{spine['match_key'].isin(set(r_fat['match_key'])).mean():.1%}")
display(r_fat.filter(regex="match_key|rest_days|games_last_7").head(3))

## 9 · Disponibilidad: pivotar por lado (solo conteos)

Disponibilidad viene en formato largo **por `side`** (una fila para el local y otra para el
visitante de cada partido) y contiene **solo conteos** de bajas (`n_missing`, `n_out`, `n_injury`,
`n_suspension`, `n_doubtful`). `reduce_availability()` la **pivota** a columnas `home_*` / `away_*`
y la puentea a `match_key` con `e2k`.

Aquí aparece un matiz de **cobertura** que reaparecerá en la sección 11: esta tabla **empieza en la
17/18**, no en la 16/17. Por eso su cobertura global es menor que la del resto; no es un error de
unión, es que **el dato no existe** para la primera temporada. Lo dejaremos reflejado y el QA lo
mide sobre el **rango activo**, no sobre todo el histórico.

In [ ]:
r_av = reduce_availability(data["av"], e2k)
comprobar(r_av, "Disponibilidad reducida (home/away)")
cob_av = spine["match_key"].isin(set(r_av["match_key"])).mean()
print("cobertura del spine:", f"{cob_av:.1%}",
      f"({len(r_av):,}/{N_PARTIDOS:,} partidos; < 100% porque arranca en 17/18, no 16/17)")
display(r_av.filter(regex="match_key|n_missing|n_injury").head(3))

## 10 · Valor de plantilla: de temporada a partido

El valor de plantilla está a nivel **club-temporada**. Para llevarlo al partido, `reduce_squad_value()`
cruza **dos veces** —una por el equipo local y otra por el visitante— sobre la pareja
`(season, team_key)`, produciendo `home_squad_value` / `away_squad_value` y sus derivados: la
diferencia, un **log-ratio** simétrico y robusto (`squad_value_logratio`, >0 favorece al local) y
la diferencia de edad media. Es un *prior* de fuerza útil sobre todo a principio de temporada,
cuando aún hay pocos partidos que alimenten al Elo.

In [12]:
# `date` viaja en la meta porque las bajas ponderadas por minutos (seccion 11)
# necesitan un instante de corte para evaluar la cuota de cada ausente.
meta = spine[["match_key","season_year","home_key","away_key","date"]].rename(
    columns={"season_year":"season"})
r_sv = reduce_squad_value(meta, data["tv"])
comprobar(r_sv, "Valor de plantilla por partido", esperado=N_PARTIDOS)
print("cobertura home_squad_value:", f"{1-r_sv['home_squad_value'].isna().mean():.1%}")
display(r_sv[["match_key","home_squad_value","away_squad_value","squad_value_logratio","age_diff"]].head(3))

[Valor de plantilla por partido]  3,800 filas × 10 cols  |  match_key único: True  |  == 3,800 partidos del spine: True
cobertura home_squad_value: 100.0%


,match_key,home_squad_value,away_squad_value,squad_value_logratio,age_diff
0,2016-08-19|deportivo_la_coruna|eibar,94.0,67.95,0.324523,0.828185
1,2016-08-19|malaga|osasuna,84.3,42.90,0.675510,0.455236
2,2016-08-20|barcelona|real_betis,765.3,84.20,2.207073,-0.528139


## 11 · Cuánto duele una baja: dos ponderaciones y la trampa del cero

Los conteos de la sección 9 dicen *cuántas* bajas hay, pero no *cuánto duelen*: no es lo mismo que
falte un suplente que el goleador. Para pesar cada baja por su **valor de mercado** cruzamos dos
tablas que hasta ahora no se habían tocado:

- el **RAW** `sofascore_missing_players.csv` — que sí trae el **nombre** del jugador ausente
  (la tabla procesada de disponibilidad solo tenía conteos), y
- **valor por jugador** de Transfermarkt (`transfermarkt_player_values`).

`build_absences()` normaliza el nombre igual que Transfermarkt (deacentuar + minúsculas + guiones),
casa por `(season, team_key, player_key)` —así un mismo apellido en otro equipo no contamina— y
agrega por partido el **valor € ausente** y su **% sobre el valor de plantilla**.

**El problema sutil —la trampa del cero.** El dato de bajas **empieza en la 17/18**. Un partido de
la 16/17 no tiene *cero bajas*: tiene **dato desconocido**. Si rellenáramos con `0` estaríamos
**fabricando** información falsa (equipos "sin bajas" que en realidad no medimos). La solución es
distinguir **dentro del universo** (temporadas con dato → un partido sin filas de baja = 0 real) de
**fuera del universo** (16/17 → se queda `NaN` = sin dato). Eso es lo que hacen las líneas de
`covered_seasons` / `in_universe` del módulo.

**La segunda medida: la cuota de minutos.** El valor de mercado dice cuánto *vale* un futbolista;
no dice cuánto *jugaba*. Un fichaje caro que no entra en las rotaciones pesa mucho en euros y poco
en el once, y al revés: un canterano barato puede ser titular indiscutible. Por eso las mismas
bajas se cruzan además contra `player_minutes.parquet` y reciben una segunda ponderación, la
**cuota de los minutos de su equipo** que el ausente venía disputando. Las dos medidas responden a
preguntas distintas y por eso conviven: cuánto capital deportivo se queda fuera y cuánto once real
se queda fuera.

**El instante de corte.** La cuota se evalúa a las **00:00 UTC del día del partido**, sobre los
minutos acumulados en encuentros **estrictamente anteriores**. Ese corte es la razón por la que la
`meta` de la celda anterior transporta ahora la columna `date`. Es deliberadamente conservador:
descarta el propio partido y cualquier otro del mismo día, de modo que la cifra sería idéntica
calculada con días de antelación. Tomar la medianoche previa en vez de la hora real de saque evita
además depender de un horario que a menudo se confirma tarde.

**Dos espacios de claves, dos cruces distintos.** El valor viaja por la normalización de
Transfermarkt (deacentuar, minúsculas, guiones) y es por tanto un cruce entre dos casas; la cuota
de minutos viaja por el `player_key` de Sofascore y es un cruce **dentro** de la misma casa, con lo
que casa de forma exacta aunque el nombre no se parezca al que publica Transfermarkt. Que un
ausente reciba cuota y no valor —o al revés— es un resultado normal, no un error.

**Suma y máximo.** Por lado se agregan la **suma** de cuotas —cuántos "onces de titular" se quedan
fuera— y el **máximo** —cuánto pesaba la baja más importante—. Solo la suma viaja como diferencia
local-visitante: es la magnitud aditiva, la que admite leerse como ventaja de un equipo sobre el
otro. Restar dos máximos compararía la peor baja de cada lado, un contraste entre dos futbolistas
concretos que dice poco del equilibrio del partido.

**La trampa del cero, un nivel más arriba.** El sub-bloque de cuotas tiene su **propio universo**
—las temporadas con minutos volcados—, que no tiene por qué coincidir con el de las bajas: un
partido puede tener parte médico y no tener minutos, o al revés, y cada bloque declara su hueco por
separado. Dentro de ese universo aún hay que distinguir un partido **sin bajas**, que pierde cuota
cero y ese cero es real, de un partido **con bajas cuyas cuotas se desconocen** —jornada 1, o un
ausente que no figura en la tabla de minutos—, donde la suma que devuelve `pandas` es `0.0` y
mentiría: no es que no se pierda nada, es que no se sabe cuánto. Ese caso se restituye a `NaN`.


In [13]:
r_abs = build_absences(data["missing"], data["pv"], meta, e2k, data["tv"],
                       player_minutes=data.get("pm"))
comprobar(r_abs, "Bajas ponderadas por valor", esperado=N_PARTIDOS)

# la trampa del cero: fuera del universo (16/17) debe quedar NaN, no 0
por_temp = r_abs.merge(meta[["match_key","season"]], on="match_key")
cob_temp = (por_temp.assign(con_dato=por_temp["home_out_value"].notna())
            .groupby("season")["con_dato"].mean())
print("cobertura de bajas€ por temporada (0% = fuera del universo, se deja NaN a propósito):")
print(cob_temp.round(2).to_string())
display(r_abs[["match_key","home_out_value","away_out_value","out_value_diff","home_out_value_pct"]].head(3))

# el sub-bloque de rol: su cobertura se reporta aparte porque tiene otro universo.
tasa = r_abs.attrs.get("minshare_rate")
print("\nbajas que recibieron cuota de minutos:",
      "sin medir (no hay tabla de minutos)" if tasa != tasa else f"{tasa:.1%}")
display(r_abs[["match_key","home_out_minshare","away_out_minshare",
               "out_minshare_diff","home_out_minshare_max"]].head(3))

[Bajas ponderadas por valor]  3,800 filas × 9 cols  |  match_key único: True  |  == 3,800 partidos del spine: True
cobertura de bajas€ por temporada (0% = fuera del universo, se deja NaN a propósito):
season
2016    0.0
2017    1.0
2018    1.0
2019    1.0
2020    1.0
2021    1.0
2022    1.0
2023    1.0
2024    1.0
2025    1.0


,match_key,home_out_value,away_out_value,out_value_diff,home_out_value_pct
0,2016-08-19|deportivo_la_coruna|eibar,NaN,NaN,NaN,NaN
1,2016-08-19|malaga|osasuna,NaN,NaN,NaN,NaN
2,2016-08-20|barcelona|real_betis,NaN,NaN,NaN,NaN


## 12 · Rotación y carga de minutos: el mismo dato agregado por equipo

La tabla de minutos sirve a un segundo consumidor. Agregada **por equipo y evento** describe cómo
gestiona el entrenador su plantilla: cuánto repite el once (`xi_continuity`), cuántos minutos
llevan encima los titulares que hoy alinea (`starters_minutes_7`, `starters_minutes_14`), cuán
concentrado está el reparto en pocos futbolistas (`minutes_hhi`) y a cuántos ha recurrido
(`n_players_used`). Es el complemento natural del bloque de fatiga, que cuenta *partidos* —un
denominador grueso, porque un suplente que entra diez minutos y un titular que juega noventa suman
lo mismo—.

**El desplazamiento hacia atrás ya viene hecho.** La alineación de un encuentro es un hecho **del
partido**: describe lo que ocurrió, igual que los goles, y usarla cruda sería fuga. Por eso
`home_xi_continuity` no es la rotación del partido que se va a predecir —que en ese momento se
desconoce— sino la que el equipo mostró en el **último que jugó**. Ese desplazamiento lo hace el
extractor, junto al dato crudo del que depende; `reduce_lineups()` se limita a cambiar la
granularidad, pivotando `event_id` + `side` a `home_*` / `away_*` y añadiendo las diferencias.

**Qué significa aquí un `NaN`.** La fuente cubre los dos lados de todos los partidos de LaLiga del
calendario, jugados o no. Un hueco no significa "no se consultó" sino "el equipo todavía no ha
jugado bastante en esta temporada": las primeras jornadas no tienen pasado del que derivar una
continuidad. El preprocesado lo marca con la misma bandera de ausencia que el resto, pero la
lectura al interpretar el modelo es distinta, y por eso el QA le concede un suelo de cobertura más
bajo que a las fuentes que deberían estar completas.


In [ ]:
if data.get("lin") is None:
    r_lin = None
    print("team_lineups.parquet no esta en data/02_processed/: el bloque de rotacion\n"
          "queda fuera del master. Para incorporarlo:\n"
          "   python scripts/extractors/lineups.py backfill && "
          "python scripts/extractors/lineups.py process")
else:
    r_lin = reduce_lineups(data["lin"], e2k)
    comprobar(r_lin, "Rotacion y minutos (home/away + diff)")
    cob_lin = spine["match_key"].isin(set(r_lin["match_key"])).mean()
    print("cobertura del spine:", f"{cob_lin:.1%}")
    display(r_lin[["match_key", "home_xi_continuity", "away_xi_continuity",
                   "xi_continuity_diff", "home_starters_minutes_7",
                   "home_minutes_hhi"]].head(3))


## 13 · Calidad de ocasión: materia prima que entra al máster sin ser variable

La última fuente que se engancha describe **la forma de la distribución de remates** de cada
equipo en cada partido: cuánto vale el disparo típico (`shot_xg_mean`), desde qué distancia se
remata (`shot_dist_mean`), qué parte del peligro nace de balón parado (`shot_setpiece_xg_pct`) y
cuánto valía la mejor ocasión (`shot_xg_max`). El máster ya sabía *cuántos* tiros hubo y *cuánto*
xG sumaron; esto es lo que ninguna de las dos cifras dice, que es cómo se reparte ese xG entre
los disparos.

**El único reductor que no necesita el puente.** Todas las piezas de la familia Sofascore hay
que traducirlas del mundo `event_id` al mundo `match_key`. Esta no: el extractor de tiros no pasa
por Sofascore y construye la clave canónica él mismo, a partir de la fecha y los dos nombres, con
el mismo `teamnames.py` que usa el resto del proyecto. Que el puente sobre y que la clave case
igualmente con la del spine es, de hecho, la mejor comprobación de que la normalización de
nombres es consistente entre fuentes independientes.

**Aquí no se calculan diferencias local–visitante**, a diferencia de la rotación. La razón no es
técnica sino de estatuto: estas columnas describen lo ocurrido en el propio partido —igual que el
marcador o el xG— y por tanto **son materia prima, no variables**. Nunca entran al modelo tal
cual; llegan a la predicción por el único camino legítimo, las ventanas retrospectivas de
`features/form.py`, que solo miran a partidos ya jugados. Calcular aquí una diferencia sería
fabricar una columna con aspecto de variable que ninguna vía debe consumir, y
`features/columns.py` la declara como fuga precisamente para cerrar esa puerta.

Es la asimetría que conviene tener presente al leer las dos últimas secciones: **la rotación llega
ya desplazada hacia atrás y pasa tal cual** al conjunto de variables; **la calidad de ocasión llega
en presente y solo se convierte en variable aguas abajo**. Misma condición de opcional, destino
opuesto.

**Qué significa aquí un `NaN`.** No es un partido sin cubrir, sino un **lado** sin distribución de
la que hablar: el equipo cuyos únicos remates fueron penaltis o goles en propia puerta no tiene
fila, porque el agregado excluye ambos por construcción y prefiere no publicar nada antes que
publicar un promedio calculado sobre lo que no es juego. Por eso un partido puede estar cubierto y
traer aun así una mitad vacía, y por eso el QA concede a este bloque un suelo de cobertura propio.


In [ ]:
if data.get("shot") is None:
    r_shot = None
    print("team_shot_quality.parquet no esta en data/02_processed/: el bloque de\n"
          "calidad de ocasion queda fuera del master. Para incorporarlo:\n"
          "   python scripts/extractors/understat_shots.py backfill && "
          "python scripts/extractors/understat_shots.py process")
else:
    r_shot = reduce_shot_quality(data["shot"])
    comprobar(r_shot, "Calidad de ocasion (home/away, sin diferencias)")
    cob_shot = spine["match_key"].isin(set(r_shot["match_key"])).mean()
    print("cobertura del spine:", f"{cob_shot:.1%}")

    # El hueco relevante no es de partido sino de LADO (ver la nota sobre el NaN).
    medio = r_shot[["home_shot_xg_mean", "away_shot_xg_mean"]].isna().any(axis=1).mean()
    print(f"partidos con un lado sin remates admisibles: {medio:.1%}")

    # Ninguna columna `*_diff`: la ausencia es deliberada y se comprueba.
    print("columnas de diferencia local-visitante:",
          [c for c in r_shot.columns if c.endswith("_diff")] or "ninguna (correcto)")
    display(r_shot[["match_key", "home_shot_xg_mean", "away_shot_xg_mean",
                    "home_shot_dist_mean", "home_shot_setpiece_xg_pct",
                    "home_shot_xg_max"]].head(3))


## 14 · Ensamblado final: left-merges sin fan-out ni colisiones

Con el spine fijado y cada fuente ya reducida a una fila por `match_key`, el ensamblado es una
secuencia de **left-merges sobre `match_key`**. Dos cuidados en cada paso:

- **Sin fan-out.** Un left-merge multiplica filas si la parte derecha tiene el `match_key`
  repetido. Por eso cada reductor garantiza `match_key` único *antes* de llegar aquí, y tras cada
  merge verificamos que `len(máster)` sigue siendo `N_PARTIDOS`.
- **Sin colisiones de columnas.** Varias fuentes traen `season` o `date`. El ayudante `_left()`
  descarta las columnas repetidas (conservando solo las del spine) para no acabar con `season_x` /
  `season_y`.

La celda replica el ensamblado merge a merge para **ver crecer el nº de columnas sin que cambie el
de filas**, y luego confirma que coincide con la función de producción `build_master()`.

In [14]:
def _left(master, part, drop=None):
    part = part.drop(columns=[x for x in (drop or []) if x in part.columns])
    dup = (set(part.columns) & set(master.columns)) - {"match_key"}
    return master.merge(part.drop(columns=list(dup)), on="match_key", how="left")

pasos = [("+ Sofascore", r_sofa, ["season","date"]),
         ("+ Fatiga", r_fat, None),
         ("+ Disponibilidad", r_av, None),
         ("+ Valor plantilla", r_sv, None),
         ("+ Bajas €", r_abs, None)]
if r_lin is not None:
    pasos.append(("+ Rotacion y minutos", r_lin, None))
if r_shot is not None:
    pasos.append(("+ Calidad de ocasion", r_shot, None))

m = spine.copy()
print(f"{'paso':<22}{'filas':>8}{'cols':>7}   sin fan-out")
print(f"{'spine (fd+xG+Elo)':<22}{len(m):>8,}{m.shape[1]:>7}")
for nombre, part, drop in pasos:
    m = _left(m, part, drop)
    print(f"{nombre:<22}{len(m):>8,}{m.shape[1]:>7}   {len(m)==N_PARTIDOS}")

# la función de producción hace exactamente esto:
m_prod = build_master(data)
print("\ncoincide con build_master():", m.shape == m_prod.shape, "| forma:", m_prod.shape)

paso                     filas   cols   sin fan-out
spine (fd+xG+Elo)        3,800     63
+ Sofascore              3,800    171   True
+ Fatiga                 3,800    183   True
+ Disponibilidad         3,800    193   True
+ Valor plantilla        3,800    202   True
+ Bajas €                3,800    210   True

coincide con build_master(): True | forma: (3800, 210)


## 15 · QA del máster: invariantes duras y cobertura por bloque

La última pieza es un **control de calidad** que convierte en *asserts* todo lo que hemos ido
comprobando a mano, para que la construcción sea reproducible y apta para CI
(`scripts/consolidate/qa_master.py`). Las invariantes:

1. **Una fila por partido** — `match_key` único.
2. **Sin fan-out** — `nº filas == nº partidos del spine`.
3. **Target completo** — `ftr` sin nulos y en `{H, D, A}`.
4. **Cobertura por bloque sobre su rango real** — cada familia (xG, Elo, Sofascore, fatiga,
   disponibilidad, valor, bajas €, bajas por rol, rotación y calidad de ocasión) se mide desde
   la temporada en que
   **empieza a existir** —disponibilidad y bajas arrancan en 17/18—, no sobre todo el histórico,
   con un suelo de cobertura propio: más exigente donde la fuente debería estar completa, más
   laxo donde el hueco es estructural y está explicado.
5. **Ninguna columna 100 % nula**, con dos excepciones declaradas: si no se ha ejecutado el
   volcado de alineaciones o el de tiros, sus columnas están vacías por diseño y el QA lo
   reporta sin dar el máster por inválido. Cualquier otra columna vacía es un merge que no ha
   casado.

La calidad de ocasión lleva además un suelo de cobertura propio, más laxo que el que se exige
a un bloque que se limita a copiar lo que la fuente publica, y la razón no es condescendencia
con el volcado: el agregado excluye penaltis y goles en propia puerta, de modo que un equipo
cuyos únicos remates fueran de esa clase no tiene distribución de la que hablar y no genera
fila. Medirlo con la vara de los demás convertiría en fallo una consecuencia declarada del
criterio de cálculo.

In [15]:
from qa_master import _BLOCKS, _BLOCKS_OPCIONALES

m = m_prod
seasons = sorted(m["season"].dropna().unique())
print("Invariantes duras")
print("  [1] match_key único      :", m["match_key"].is_unique)
print("  [2] sin fan-out          :", len(m) == N_PARTIDOS, f"({len(m):,} filas)")
print("  [3] ftr completo y válido:", (not m["ftr"].isna().any())
      and set(m["ftr"].unique()) <= {"H","D","A"})
allnull = [col for col in m.columns if m[col].isna().all()]
print("  [5] columnas 100% nulas  :", len(allnull), allnull[:5])

# los bloques opcionales solo se miden si traen dato: si el volcado de
# alineaciones no se ha ejecutado, su ausencia es un hueco declarado.
vivos = {n: c for n, c in _BLOCKS_OPCIONALES.items()
         if any(x in m.columns and not m[x].isna().all() for x in c)}

print("\n[4] Cobertura por bloque sobre su rango activo:")
for nombre, cols in {**_BLOCKS, **vivos}.items():
    pres = [x for x in cols if x in m.columns]
    per = {s: 1 - m.loc[m["season"]==s, pres].isna().mean().mean() for s in seasons}
    activo = [s for s in seasons if per[s] > 0]
    cov = np.mean([per[s] for s in activo]) if activo else float("nan")
    nota = "" if activo and activo[0]==seasons[0] else f"   <- empieza en {activo[0] if activo else '?'}"
    print(f"  {nombre:16s}: {cov:6.1%}{nota}")

Invariantes duras
  [1] match_key único      : True
  [2] sin fan-out          : True (3,800 filas)
  [3] ftr completo y válido: True
  [5] columnas 100% nulas  : 0 []

[4] Cobertura por bloque sobre su rango activo:
  xg              : 100.0%
  elo             : 100.0%
  sofa_rating     : 100.0%
  fatiga          : 100.0%
  disponibilidad  :  93.6%   <- empieza en 2017/18
  valor_plantilla : 100.0%
  bajas_euros     : 100.0%   <- empieza en 2017/18


## 16 · Guardado y siguiente paso

`build_master.py` escribe el máster en `data/02_processed/master_matches.parquet` de forma
**atómica** (temporal + `rename`, para no dejar nunca un parquet a medias). Desde la línea de
comandos:

```bash
python scripts/consolidate/build_master.py        # construye, valida y guarda
python scripts/consolidate/build_master.py --dry  # construye y solo reporta (no escribe)
python scripts/consolidate/qa_master.py           # QA completo (sale !=0 si algo falla)
```

Con el máster —**una fila por partido de LaLiga, 16/17 en adelante, con target, mercado, xG, Elo,
estadística Sofascore, fatiga, disponibilidad, valor de plantilla, bajas ponderadas por valor y por
minutos, rotación del once y la forma de la distribución de remates —esta última como materia
prima, no como variable—** cerramos la consolidación. El siguiente notebook parte de esta tabla para construir las **variables del
modelo** (formas, medias móviles, diferencias local-visitante) y, sobre ellas, el clasificador 1X2.